In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

In [ ]:
import os
import time
import random
import warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.ticker as mticker
from matplotlib.colors import LinearSegmentedColormap   # single top-level import (fix: duplicate imports)
import seaborn as sns
import tensorflow as tf

from PIL import Image, UnidentifiedImageError
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    roc_curve
)

warnings.filterwarnings('ignore')

# FancyBboxPatch, Counter, Path, bare `import matplotlib` — all removed (fix: dead imports)

from tensorflow.keras.applications import (
    MobileNetV2, ResNet50, EfficientNetB0, VGG16, InceptionV3
)
from tensorflow.keras.applications.mobilenet_v2 import preprocess_input as mobilenet_preprocess
from tensorflow.keras.applications.resnet50     import preprocess_input as resnet_preprocess
from tensorflow.keras.applications.efficientnet import preprocess_input as efficientnet_preprocess
from tensorflow.keras.applications.vgg16        import preprocess_input as vgg_preprocess
from tensorflow.keras.applications.inception_v3 import preprocess_input as inception_preprocess
from tensorflow.keras import models
from tensorflow.keras.layers import (
    Dense, Dropout, GlobalAveragePooling2D,
    Input, BatchNormalization,
    RandomFlip, RandomRotation, RandomContrast, RandomZoom
)


# ============================================================
# GLOBAL PLOT STYLE — light professional theme
# ============================================================

plt.rcParams.update({
    'figure.facecolor':   '#ffffff',
    'figure.dpi':         120,
    'axes.facecolor':     '#f6f8fa',
    'axes.edgecolor':     '#d0d7de',
    'axes.labelcolor':    '#24292f',
    'axes.titlecolor':    '#1f2328',
    'axes.titlesize':     13,
    'axes.titleweight':   'bold',
    'axes.titlepad':      10,
    'axes.labelsize':     11,
    'axes.labelpad':      6,
    'axes.grid':          True,
    'axes.spines.top':    False,
    'axes.spines.right':  False,
    'grid.color':         '#d8dee4',
    'grid.linewidth':     0.7,
    'grid.alpha':         0.8,
    'xtick.color':        '#57606a',
    'ytick.color':        '#57606a',
    'xtick.labelsize':    9,
    'ytick.labelsize':    9,
    'xtick.major.pad':    4,
    'ytick.major.pad':    4,
    'legend.facecolor':   '#ffffff',
    'legend.edgecolor':   '#d0d7de',
    'legend.labelcolor':  '#24292f',
    'legend.fontsize':    9,
    'legend.framealpha':  0.9,
    'legend.borderpad':   0.6,
    'text.color':         '#24292f',
    'figure.titlesize':   15,
    'figure.titleweight': 'bold',
    'lines.linewidth':    2.2,
    'lines.antialiased':  True,
    'patch.antialiased':  True,
    'font.family':        'DejaVu Sans',
    'savefig.facecolor':  '#ffffff',
    'savefig.edgecolor':  'none',
    'savefig.dpi':        150,
})

MODEL_PALETTE = {
    'MobileNetV2':    '#1f6feb',
    'ResNet50':       '#cf222e',
    'EfficientNetB0': '#1a7f37',
    'VGG16':          '#8250df',
    'InceptionV3':    '#d4770a',
}
CLASS_COLORS = {
    'NORMAL':    '#1a7f37',
    'PNEUMONIA': '#cf222e',
}
METRIC_COLORS = ['#1f6feb', '#1a7f37', '#d4770a', '#8250df', '#cf222e']


# ============================================================
# SECTION 2 : GPU SETUP
# ============================================================

gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        # set_memory_growth moved out of .experimental. in TF 2.1+
        # but some Kaggle/Colab builds still ship the older path — try both
        try:
            tf.config.set_memory_growth(gpu, True)
        except AttributeError:
            tf.config.experimental.set_memory_growth(gpu, True)
    print(f"GPUs Available : {len(gpus)}")
else:
    print("No GPU found — running on CPU.")

print(f"TensorFlow Version : {tf.__version__}")


# ============================================================
# SECTION 3 : REPRODUCIBILITY
# ============================================================

SEED = 42

os.environ['PYTHONHASHSEED']         = str(SEED)
os.environ['TF_DETERMINISTIC_OPS']   = '1'
os.environ['TF_CUDNN_DETERMINISTIC'] = '1'

random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

print(f"Global random seed locked to {SEED}  (Python / NumPy / TensorFlow)")


# ============================================================
# SECTION 4 : DOWNLOAD DATASET
# ============================================================

!kaggle datasets download -d paultimothymooney/chest-xray-pneumonia -q
!unzip -q chest-xray-pneumonia.zip


# ============================================================
# SECTION 5 : PATHS & GLOBAL PARAMETERS
# ============================================================

DATASET_PATH  = "chest_xray/chest_xray"
TRAIN_PATH    = os.path.join(DATASET_PATH, "train")
VAL_PATH      = os.path.join(DATASET_PATH, "val")
TEST_PATH     = os.path.join(DATASET_PATH, "test")

IMG_SIZE      = 224
BATCH_SIZE    = 32
AUTOTUNE      = tf.data.AUTOTUNE

EPOCHS_PHASE1 = 15   # frozen head
EPOCHS_PHASE2 = 10   # fine-tune top layers
               # Total = 25 epochs per model

N_METRICS     = 5    # fix: define at module level so Section 20 dashboard
                     # doesn't depend on Section 17 having run first

CLASSES       = ['NORMAL', 'PNEUMONIA']


# ============================================================
# SECTION 6 : EXPLORATORY DATA ANALYSIS (EDA)
# ============================================================

print("\n" + "=" * 70)
print("EXPLORATORY DATA ANALYSIS")
print("=" * 70)

# ── 6.1  Image counts ────────────────────────────────────────────────────

def count_images(split_path):
    counts = {}
    for cls in CLASSES:
        folder = os.path.join(split_path, cls)
        if os.path.exists(folder):
            counts[cls] = len([
                f for f in os.listdir(folder)
                if f.lower().endswith(('.png', '.jpg', '.jpeg'))
            ])
        else:
            counts[cls] = 0
    return counts

train_counts = count_images(TRAIN_PATH)
val_counts   = count_images(VAL_PATH)
test_counts  = count_images(TEST_PATH)

splits = ['Train', 'Validation', 'Test']
counts = [train_counts, val_counts, test_counts]

eda_rows = []
for split, cnt in zip(splits, counts):
    for cls in CLASSES:
        eda_rows.append({'Split': split, 'Class': cls, 'Count': cnt[cls]})

eda_df = pd.DataFrame(eda_rows)
pivot  = eda_df.pivot(index='Split', columns='Class', values='Count')
pivot['Total']       = pivot.sum(axis=1)
pivot['Pneumonia %'] = (pivot['PNEUMONIA'] / pivot['Total'] * 100).round(1)
print("\nImage Count Summary\n" + "-" * 50)
print(pivot)

# ── 6.2  Class distribution bar chart ────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Class Distribution Across Splits", fontsize=15,
             fontweight='bold', color='#1f2328', y=1.02)

for ax, split, cnt in zip(axes, splits, counts):
    vals = [cnt[c] for c in CLASSES]
    cols = [CLASS_COLORS[c] for c in CLASSES]
    bars = ax.bar(CLASSES, vals, color=cols, edgecolor='white',
                  linewidth=0.8, width=0.50, alpha=0.85, zorder=3)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + max(vals) * 0.02,
                f"{bar.get_height():,}",
                ha='center', va='bottom', fontsize=10,
                fontweight='bold', color='#24292f')
    ax.set_title(split, pad=10)
    ax.set_ylabel("Number of Images")
    ax.set_ylim(0, max(vals) * 1.22)
    ax.tick_params(axis='x', labelsize=10)

plt.tight_layout()
plt.savefig("eda_class_distribution.png", bbox_inches='tight')
plt.show()

# ── 6.3  Pie charts ──────────────────────────────────────────────────────

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
fig.suptitle("Class Balance per Split", fontsize=15,
             fontweight='bold', color='#1f2328', y=1.02)
pie_colors = [CLASS_COLORS['NORMAL'], CLASS_COLORS['PNEUMONIA']]

for ax, split, cnt in zip(axes, splits, counts):
    values = [cnt[c] for c in CLASSES]
    wedges, texts, autotexts = ax.pie(
        values, labels=CLASSES, colors=pie_colors,
        autopct='%1.1f%%', startangle=90,
        wedgeprops={'edgecolor': '#ffffff', 'linewidth': 2.0},
        textprops={'color': '#24292f', 'fontsize': 10}
    )
    for at in autotexts:
        at.set_fontweight('bold')
        at.set_fontsize(10)
    ax.set_title(split, pad=10)

plt.tight_layout()
plt.savefig("eda_pie_charts.png", bbox_inches='tight')
plt.show()

# ── 6.4  Sample images grid (crisp, native resolution) ───────────────────

def load_sample_images(folder, n=5):
    """
    Load n images at NATIVE resolution as grayscale numpy arrays.
    No resizing, no thumbnail — full pixel fidelity preserved.
    """
    image_files = [f for f in os.listdir(folder)
                   if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    imgs = []
    for f in sorted(image_files)[:n]:
        try:
            img = Image.open(os.path.join(folder, f)).convert('L')
            imgs.append(np.array(img))
        except Exception as e:
            print(f"Skipping {f}: {e}")
    return imgs

fig, axes = plt.subplots(2, 5, figsize=(20, 8))
fig.suptitle("Sample Chest X-Ray Images", fontsize=15, fontweight='bold',
             color='#1f2328', y=1.02)

for row, cls in enumerate(CLASSES):
    samples = load_sample_images(os.path.join(TRAIN_PATH, cls), n=5)
    for col in range(5):
        ax = axes[row, col]
        ax.set_facecolor('#ffffff')
        if col < len(samples):
            # interpolation='none': zero resampling — sharpest possible render
            ax.imshow(samples[col], cmap='gray', interpolation='none',
                      vmin=0, vmax=255)
        ax.axis('off')
        label       = cls if col == 0 else f"#{col + 1}"
        label_color = CLASS_COLORS[cls] if col == 0 else '#57606a'
        fw          = 'bold' if col == 0 else 'normal'
        ax.set_title(label, fontsize=9, fontweight=fw, color=label_color, pad=4)

plt.tight_layout()
plt.savefig("eda_sample_images.png", bbox_inches='tight')
plt.show()

# ── 6.5  Image size distribution ─────────────────────────────────────────

def get_image_sizes(folder, max_images=200):
    widths, heights = [], []
    image_files = [f for f in os.listdir(folder)
                   if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    for f in image_files[:max_images]:
        try:
            img = Image.open(os.path.join(folder, f))
            w, h = img.size
            widths.append(w)
            heights.append(h)
        except Exception:
            pass
    return widths, heights

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
fig.suptitle("Image Size Distribution (Training Set, 200 samples/class)",
             fontsize=15, fontweight='bold', color='#1f2328', y=1.02)

for cls in CLASSES:
    ws, hs = get_image_sizes(os.path.join(TRAIN_PATH, cls))
    c = CLASS_COLORS[cls]
    axes[0].hist(ws, bins=35, alpha=0.75, label=cls, color=c, edgecolor='white',
                 linewidth=0.5, zorder=3)
    axes[1].hist(hs, bins=35, alpha=0.75, label=cls, color=c, edgecolor='white',
                 linewidth=0.5, zorder=3)

for ax, title in zip(axes, ['Width Distribution', 'Height Distribution']):
    ax.set_title(title, pad=10)
    ax.set_xlabel("Pixels")
    ax.set_ylabel("Frequency")
    ax.legend()

plt.tight_layout()
plt.savefig("eda_size_distribution.png", bbox_inches='tight')
plt.show()

# ── 6.6  Pixel intensity distribution ────────────────────────────────────

def mean_pixel_intensities(folder, max_images=200):
    means = []
    image_files = [f for f in os.listdir(folder)
                   if f.lower().endswith(('.png', '.jpg', '.jpeg'))]
    for f in image_files[:max_images]:
        try:
            img = np.array(Image.open(os.path.join(folder, f)).convert('L'))
            means.append(img.mean())
        except Exception:
            pass
    return np.array(means)

fig, ax = plt.subplots(figsize=(14, 5))
fig.suptitle("Mean Pixel Intensity Distribution (Grayscale)",
             fontsize=15, fontweight='bold', color='#1f2328', y=1.02)

for cls in CLASSES:
    intensities = mean_pixel_intensities(os.path.join(TRAIN_PATH, cls))
    ax.hist(intensities, bins=35, alpha=0.75, label=cls,
            color=CLASS_COLORS[cls], edgecolor='white', linewidth=0.5, zorder=3)

ax.set_xlabel("Mean Pixel Value (0–255)")
ax.set_ylabel("Frequency")
ax.legend()
plt.tight_layout()
plt.savefig("eda_pixel_intensity.png", bbox_inches='tight')
plt.show()

# ── 6.7  Representative images per class (replaces blurry average) ───────
#
# WHY the average image is always blurry:
#   Pixel-averaging 300 X-rays mathematically cancels any detail that
#   differs across patients (anatomy size, rotation, chest shape).
#   The blur is in the data — no code fix can remove it.
#
# BETTER: show the 3 sharpest real images per class.
#   "Sharpest" = highest Laplacian variance (standard focus metric).
#   These are actual X-rays, not a synthetic average, so they are crisp.

def pick_sharpest_images(folder, n=3, size=(512, 512)):
    """
    Return the n images with the highest Laplacian variance (sharpness score)
    from a folder, loaded as uint8 numpy arrays at `size` resolution.

    Laplacian variance is the standard no-reference sharpness metric:
    a blurry image has low variance (smooth gradients), a sharp image
    has high variance (strong edges).
    """
    from PIL import ImageFilter

    image_files = [f for f in os.listdir(folder)
                   if f.lower().endswith(('.png', '.jpg', '.jpeg'))]

    scored = []
    for f in image_files:
        try:
            pil_img = Image.open(os.path.join(folder, f)).convert('L')
            # Compute sharpness on a small thumbnail (fast)
            thumb = pil_img.resize((128, 128), Image.LANCZOS)
            arr   = np.array(thumb, dtype=np.float32)
            # Laplacian via simple finite difference
            lap   = (np.roll(arr, 1, 0) + np.roll(arr, -1, 0) +
                     np.roll(arr, 1, 1) + np.roll(arr, -1, 1) - 4 * arr)
            score = lap.var()
            # Load display version at target size
            disp  = np.array(pil_img.resize(size, Image.LANCZOS), dtype=np.uint8)
            scored.append((score, disp, f))
        except Exception:
            pass

    scored.sort(key=lambda t: t[0], reverse=True)
    return [(disp, fname) for _, disp, fname in scored[:n]]

fig, axes = plt.subplots(2, 3, figsize=(18, 12))
fig.patch.set_facecolor('#ffffff')
fig.suptitle("Representative X-Ray Images per Class  (Top 3 Sharpest)",
             fontsize=15, fontweight='bold', color='#1f2328', y=1.02)

for row, cls in enumerate(CLASSES):
    sharp_imgs = pick_sharpest_images(os.path.join(TRAIN_PATH, cls), n=3)
    label_color = CLASS_COLORS[cls]

    for col, (img_arr, fname) in enumerate(sharp_imgs):
        ax = axes[row, col]
        ax.set_facecolor('#ffffff')
        ax.imshow(img_arr, cmap='gray', interpolation='none',
                  vmin=0, vmax=255)
        ax.axis('off')

        rank_label = f"{'NORMAL' if cls == 'NORMAL' else 'PNEUMONIA'}  #{col + 1}"
        ax.set_title(rank_label, fontsize=10, fontweight='bold',
                     color=label_color, pad=7)

    # Row label on the leftmost axes
    axes[row, 0].text(-0.04, 0.5, cls,
                      transform=axes[row, 0].transAxes,
                      fontsize=13, fontweight='bold', color=label_color,
                      va='center', ha='right', rotation=90)

plt.tight_layout(rect=[0.03, 0, 1, 0.96])
plt.savefig("eda_representative_images.png", bbox_inches='tight')
plt.show()

print("\nEDA Complete ✓")


# ============================================================
# SECTION 7 : DATA AUGMENTATION
# ============================================================

data_augmentation = models.Sequential([
    RandomFlip("horizontal", seed=SEED),
    RandomRotation(0.1,      seed=SEED),
    RandomContrast(0.1,      seed=SEED),
    RandomZoom(0.1,          seed=SEED)
], name="data_augmentation")


# ============================================================
# SECTION 8 : LOAD DATASETS
# ============================================================

train_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    validation_split=0.2,
    subset='training',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

val_dataset = tf.keras.utils.image_dataset_from_directory(
    TRAIN_PATH,
    validation_split=0.2,
    subset='validation',
    seed=SEED,
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

test_dataset = tf.keras.utils.image_dataset_from_directory(
    TEST_PATH,
    shuffle=False,                         # must stay False — label order relied on below
    image_size=(IMG_SIZE, IMG_SIZE),
    batch_size=BATCH_SIZE,
    label_mode='binary'
)

# ── Cache test labels ONCE here, outside the training loop ───────────────
# fix: avoids re-iterating raw test_dataset 5 times (once per model)
true_classes = np.concatenate(
    [y.numpy() for _, y in test_dataset], axis=0
).astype(int)
print(f"\nTest set: {len(true_classes)} samples  "
      f"| NORMAL={np.sum(true_classes==0)}  PNEUMONIA={np.sum(true_classes==1)}")


# ============================================================
# SECTION 9 : CLASS WEIGHTS
# ============================================================

normal_count    = len([f for f in os.listdir(os.path.join(TRAIN_PATH, 'NORMAL'))
                       if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
pneumonia_count = len([f for f in os.listdir(os.path.join(TRAIN_PATH, 'PNEUMONIA'))
                       if f.lower().endswith(('.png', '.jpg', '.jpeg'))])
total_train     = normal_count + pneumonia_count

class_weight = {
    0: total_train / (2 * normal_count),
    1: total_train / (2 * pneumonia_count)
}
print(f"Class weights → NORMAL: {class_weight[0]:.3f} | PNEUMONIA: {class_weight[1]:.3f}")


# ============================================================
# SECTION 10 : PREPROCESSING PIPELINE FACTORY
# ============================================================

def create_pipeline(dataset, preprocess_fn, augment=False):
    def process(image, label):
        if augment:
            image = data_augmentation(image, training=True)
        image = preprocess_fn(image)
        return image, label
    return (
        dataset
        .map(process, num_parallel_calls=AUTOTUNE)
        .prefetch(AUTOTUNE)
    )


# ============================================================
# SECTION 11 : MODEL CONFIGURATIONS
# ============================================================

model_configs = {
    'MobileNetV2':    {'model_fn': MobileNetV2,    'preprocess_fn': mobilenet_preprocess},
    'ResNet50':       {'model_fn': ResNet50,        'preprocess_fn': resnet_preprocess},
    'EfficientNetB0': {'model_fn': EfficientNetB0,  'preprocess_fn': efficientnet_preprocess},
    'VGG16':          {'model_fn': VGG16,           'preprocess_fn': vgg_preprocess},
    'InceptionV3':    {'model_fn': InceptionV3,     'preprocess_fn': inception_preprocess},
}


# ============================================================
# SECTION 12 : MODEL BUILDER
# ============================================================

def build_model(model_class, model_name):
    tf.random.set_seed(SEED)
    base_model = model_class(
        weights='imagenet',
        include_top=False,
        input_shape=(IMG_SIZE, IMG_SIZE, 3)
    )
    base_model.trainable = False

    _ki     = tf.keras.initializers.GlorotUniform(seed=SEED)
    inputs  = Input(shape=(IMG_SIZE, IMG_SIZE, 3))
    x       = base_model(inputs, training=False)
    x       = GlobalAveragePooling2D()(x)
    x       = Dense(256, activation='relu', kernel_initializer=_ki)(x)
    x       = BatchNormalization()(x)
    x       = Dropout(0.4, seed=SEED)(x)
    x       = Dense(128, activation='relu', kernel_initializer=_ki)(x)
    x       = Dropout(0.2, seed=SEED)(x)
    outputs = Dense(1, activation='sigmoid', kernel_initializer=_ki, dtype='float32')(x)
    return models.Model(inputs, outputs, name=model_name)


# ============================================================
# SECTION 13 : MODEL ARCHITECTURE DISPLAY
# ============================================================

def print_model_architecture(model, model_name):
    print("\n" + "=" * 70)
    print(f"Model Architecture — {model_name}")
    print("=" * 70)

    summary_lines = []
    # fix: show_trainable added in Keras 2.9 — wrapped in try/except for compat
    try:
        model.summary(
            print_fn=lambda line: summary_lines.append(line),
            expand_nested=False,
            show_trainable=True
        )
    except TypeError:
        model.summary(
            print_fn=lambda line: summary_lines.append(line),
            expand_nested=False
        )
    for line in summary_lines:
        print(line)

    txt_path = f"architecture_{model_name}.txt"
    with open(txt_path, 'w') as f:
        f.write(f"Model Architecture — {model_name}\n\n")
        f.write('\n'.join(summary_lines))
    print(f"\n  Architecture saved → {txt_path}")

    # ── Matplotlib architecture table ─────────────────────────────────────
    col_labels = ['Layer (type)', 'Output Shape', 'Param #', 'Connected to']
    col_widths = [0.30, 0.25, 0.18, 0.27]
    rows = []
    for layer in model.layers:
        ltype = layer.__class__.__name__
        lname = layer.name
        try:
            shape = str(layer.output_shape)
        except AttributeError:
            shape = 'multiple'
        params = f"{layer.count_params():,}"
        try:
            inbound = layer._inbound_nodes
            connected = ', '.join(
                n.inbound_layers.name
                if not isinstance(n.inbound_layers, list)
                else ', '.join(l.name for l in n.inbound_layers)
                for n in inbound[:1]
            ) if inbound else '-'
        except Exception:
            connected = '-'
        rows.append([
            f"{lname}\n({ltype})",
            shape,
            params,
            connected[:28] + '…' if len(connected) > 28 else connected
        ])

    n_rows  = len(rows)
    fig_h   = max(4, 0.48 * n_rows + 2.8)
    fig, ax = plt.subplots(figsize=(14, fig_h), facecolor='#ffffff')
    ax.set_facecolor('#ffffff')
    ax.axis('off')
    fig.suptitle(f"Model Architecture — {model_name}",
                 fontsize=15, fontweight='bold', color='#1f2328', y=0.99)

    tbl = ax.table(
        cellText  = rows,
        colLabels = col_labels,
        cellLoc   = 'left',
        loc       = 'center',
        colWidths = col_widths
    )
    tbl.auto_set_font_size(False)
    tbl.set_fontsize(8.5)
    tbl.scale(1, 1.65)

    hc = MODEL_PALETTE.get(model_name, '#1f6feb')
    for ci in range(len(col_labels)):
        cell = tbl[0, ci]
        cell.set_facecolor(hc)
        cell.set_text_props(color='#ffffff', fontweight='bold')
        cell.set_edgecolor('#ffffff')

    for ri in range(1, n_rows + 1):
        bg = '#f6f8fa' if ri % 2 == 0 else '#ffffff'
        for ci in range(len(col_labels)):
            c = tbl[ri, ci]
            c.set_facecolor(bg)
            c.set_text_props(color='#24292f')
            c.set_edgecolor('#d0d7de')
        tbl[ri, 2].set_text_props(ha='right', color=hc)

    # fix: use np.prod(w.shape) for param count — consistent with Keras conventions
    total_params     = model.count_params()
    trainable_params = int(np.sum([np.prod(w.shape) for w in model.trainable_weights]))
    non_trainable    = total_params - trainable_params

    def _fmt(n):
        return f"{n:,}  ({n * 4 / 1024**2:.2f} MB)"

    footer = (
        f"Total params: {_fmt(total_params)}\n"
        f"Trainable:    {_fmt(trainable_params)}\n"
        f"Frozen:       {_fmt(non_trainable)}"
    )
    fig.text(0.01, 0.01, footer, fontsize=8.5, family='monospace',
             color='#57606a', va='bottom')

    plt.tight_layout(rect=[0, 0.07, 1, 0.97])
    save_path = f"architecture_{model_name}.png"
    plt.savefig(save_path, bbox_inches='tight')
    plt.show()
    print(f"  Architecture figure saved → {save_path}")


# ============================================================
# SECTION 14 : TRAINING HISTORY PLOTTER
# ============================================================

def _ema(values, alpha=0.3):
    """Exponential moving average — smooth noisy training curves."""
    s, smoothed = values[0], []
    for v in values:
        s = alpha * v + (1 - alpha) * s
        smoothed.append(s)
    return smoothed

def plot_training_history(history_p1, history_p2, model_name):
    acc      = history_p1.history['accuracy']     + history_p2.history['accuracy']
    val_acc  = history_p1.history['val_accuracy'] + history_p2.history['val_accuracy']
    loss     = history_p1.history['loss']         + history_p2.history['loss']
    val_loss = history_p1.history['val_loss']     + history_p2.history['val_loss']

    n_epochs     = len(acc)
    epochs_range = range(1, n_epochs + 1)
    p2_start     = EPOCHS_PHASE1
    mc           = MODEL_PALETTE.get(model_name, '#58a6ff')

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5), facecolor='#ffffff')
    fig.suptitle(f"{model_name}  ·  Training History  ({n_epochs} epochs)",
                 fontsize=15, fontweight='bold', color='#1f2328', y=1.02)

    for ax, raw_train, raw_val, label_y in [
        (ax1, acc,  val_acc,  'Accuracy'),
        (ax2, loss, val_loss, 'Loss')
    ]:
        ax.set_facecolor('#f6f8fa')

        # Faded raw lines + bold EMA lines
        ax.plot(epochs_range, raw_train, color=mc,        alpha=0.22, linewidth=1.2)
        ax.plot(epochs_range, raw_val,   color='#d4770a', alpha=0.22, linewidth=1.2)
        ax.plot(epochs_range, _ema(raw_train), color=mc,
                linewidth=2.5, label=f'Train {label_y}', zorder=5)
        ax.plot(epochs_range, _ema(raw_val), color='#d4770a',
                linewidth=2.5, linestyle='--', label=f'Val {label_y}', zorder=5)

        # Phase 2 shading and boundary line
        ax.axvspan(p2_start + 0.5, n_epochs + 0.5, color='#000000', alpha=0.04, zorder=1)
        ax.axvline(p2_start + 0.5, color='#cf222e', linewidth=1.5,
                   linestyle=':', alpha=0.9, label='Fine-tuning starts', zorder=6)

        # fix: set explicit xlim BEFORE reading ylim so annotations land correctly
        ax.set_xlim(1, n_epochs)
        ax.autoscale(axis='y')
        ymin, ymax = ax.get_ylim()
        y_range    = ymax - ymin

        # Best val annotation
        if label_y == 'Accuracy':
            best_ep  = int(np.argmax(raw_val)) + 1
            best_val = max(raw_val)
            ann_y    = best_val - y_range * 0.08
        else:
            best_ep  = int(np.argmin(raw_val)) + 1
            best_val = min(raw_val)
            ann_y    = best_val + y_range * 0.08

        ann_x = best_ep + max(1, n_epochs * 0.08)
        if ann_x > n_epochs * 0.9:
            ann_x = best_ep - max(1, n_epochs * 0.15)

        label_str = f'Best val: {best_val:.4f}\n@ epoch {best_ep}' \
                    if label_y == 'Accuracy' else \
                    f'Min val: {best_val:.4f}\n@ epoch {best_ep}'

        ax.annotate(label_str,
                    xy=(best_ep, best_val),
                    xytext=(ann_x, ann_y),
                    color='#d4770a', fontsize=8.5, fontweight='bold',
                    arrowprops=dict(arrowstyle='->', color='#d4770a', lw=1.5))

        ax.set_title(label_y, fontsize=13, fontweight='bold', color='#1f2328', pad=8)
        ax.set_xlabel("Epoch", fontsize=11)
        ax.set_ylabel(label_y, fontsize=11)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(integer=True))
        ax.legend(loc='best', framealpha=0.85)

        # Phase region labels — drawn after ylim is final
        mid1 = p2_start / 2 + 0.5
        mid2 = p2_start + (n_epochs - p2_start) / 2 + 0.5
        ax.text(mid1, ymin + y_range * 0.04,
                'Phase 1\n(frozen)', ha='center', va='bottom',
                color='#57606a', fontsize=8, fontstyle='italic')
        ax.text(mid2, ymin + y_range * 0.04,
                'Phase 2\n(fine-tune)', ha='center', va='bottom',
                color='#57606a', fontsize=8, fontstyle='italic')

    plt.tight_layout()
    plt.savefig(f"history_{model_name}.png", bbox_inches='tight')
    plt.show()


# ============================================================
# SECTION 15 : CONFUSION MATRIX PLOTTER
# ============================================================

def plot_confusion_matrix(cm, model_name):
    mc  = MODEL_PALETTE.get(model_name, '#1f6feb')
    cmap = LinearSegmentedColormap.from_list('model_cm', ['#f6f8fa', mc], N=256)

    fig, ax = plt.subplots(figsize=(6, 5), facecolor='#ffffff')
    ax.set_facecolor('#f6f8fa')

    sns.heatmap(
        cm, annot=True, fmt='d', cmap=cmap,
        xticklabels=CLASSES, yticklabels=CLASSES,
        linewidths=1.2, linecolor='#ffffff', ax=ax,
        annot_kws={'size': 15, 'weight': 'bold', 'color': '#1f2328'},
        cbar_kws={'shrink': 0.8}
    )
    ax.set_title(f"{model_name}  ·  Confusion Matrix",
                 fontsize=13, fontweight='bold', color='#1f2328', pad=10)
    ax.set_xlabel("Predicted Label", fontsize=11, color='#24292f')
    ax.set_ylabel("True Label",      fontsize=11, color='#24292f')
    ax.tick_params(colors='#24292f', labelsize=10)
    ax.figure.axes[-1].yaxis.set_tick_params(color='#57606a', labelcolor='#57606a')

    acc = (cm[0, 0] + cm[1, 1]) / cm.sum()
    fig.text(0.5, 0.01, f"Overall Accuracy: {acc:.4f}",
             ha='center', fontsize=10, color='#57606a')

    plt.tight_layout(rect=[0, 0.04, 1, 1])
    plt.savefig(f"cm_{model_name}.png", bbox_inches='tight')
    plt.show()


# ============================================================
# SECTION 16 : TRAIN + EVALUATE ALL MODELS
# ============================================================

evaluation_results = {}
comparison_data    = []
all_roc_data       = {}

for idx, (model_name, config) in enumerate(model_configs.items(), 1):

    print("\n" + "=" * 70)
    print(f"[{idx}/5] TRAINING : {model_name}")
    print("=" * 70)

    start_time = time.time()

    # Build
    model = build_model(config['model_fn'], model_name)
    print_model_architecture(model, model_name)

    # Pipelines
    tf.random.set_seed(SEED)
    np.random.seed(SEED)
    train_ds = create_pipeline(train_dataset, config['preprocess_fn'], augment=True)
    val_ds   = create_pipeline(val_dataset,   config['preprocess_fn'], augment=False)
    test_ds  = create_pipeline(test_dataset,  config['preprocess_fn'], augment=False)

    # ── Phase 1 : frozen backbone ─────────────────────────────────────────
    print(f"\n  Phase 1 — Training classification head ({EPOCHS_PHASE1} epochs) ...")
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-3),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    history_phase1 = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS_PHASE1, class_weight=class_weight, verbose=1
    )

    # ── Phase 2 : fine-tune top 20 layers ────────────────────────────────
    print(f"\n  Phase 2 — Fine-tuning top layers ({EPOCHS_PHASE2} epochs) ...")
    tf.random.set_seed(SEED)

    # fix: look up backbone by name, not hardcoded index [1]
    base_model = next(l for l in model.layers
                      if isinstance(l, tf.keras.Model) and l.name != model_name)
    base_model.trainable = True
    for layer in base_model.layers[:-20]:
        layer.trainable = False

    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),
        loss='binary_crossentropy',
        metrics=['accuracy', tf.keras.metrics.AUC(name='auc')]
    )
    history_phase2 = model.fit(
        train_ds, validation_data=val_ds,
        epochs=EPOCHS_PHASE2, class_weight=class_weight, verbose=1
    )

    plot_training_history(history_phase1, history_phase2, model_name)

    # ── Evaluate ──────────────────────────────────────────────────────────
    predictions_prob  = model.predict(test_ds, verbose=0).reshape(-1)
    predicted_classes = (predictions_prob > 0.5).astype(int)
    # true_classes already extracted once above Section 9 (fix: no repeated iteration)

    accuracy  = accuracy_score(true_classes, predicted_classes)
    precision = precision_score(true_classes, predicted_classes)
    recall    = recall_score(true_classes, predicted_classes)
    f1        = f1_score(true_classes, predicted_classes)
    roc_auc   = roc_auc_score(true_classes, predictions_prob)
    cm        = confusion_matrix(true_classes, predicted_classes)
    fpr, tpr, _ = roc_curve(true_classes, predictions_prob)
    elapsed   = time.time() - start_time

    evaluation_results[model_name] = {
        'accuracy': accuracy, 'precision': precision,
        'recall': recall, 'f1': f1, 'roc_auc': roc_auc
    }
    comparison_data.append([model_name, accuracy, precision, recall, f1, roc_auc])
    all_roc_data[model_name] = (fpr, tpr, roc_auc)

    print(f"\n  ✓ {model_name}  |  Acc: {accuracy:.4f}  |  "
          f"F1: {f1:.4f}  |  AUC: {roc_auc:.4f}  |  {elapsed:.1f}s")
    print(classification_report(true_classes, predicted_classes, target_names=CLASSES))

    plot_confusion_matrix(cm, model_name)

    # fix: del model is sufficient; clear_session already frees GPU memory
    tf.keras.backend.clear_session()
    del model


# ============================================================
# SECTION 17 : FINAL COMPARISON TABLE
# ============================================================

comparison_df = pd.DataFrame(
    comparison_data,
    columns=['Model', 'Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']
)
comparison_df = comparison_df.sort_values('ROC AUC', ascending=False).reset_index(drop=True)
comparison_df.index += 1

print("\n" + "=" * 70)
print("FINAL MODEL COMPARISON")
print("=" * 70)
print(comparison_df.to_string())


# ============================================================
# SECTION 18 : GROUPED BAR CHART — ALL METRICS
# ============================================================

metrics    = ['Accuracy', 'Precision', 'Recall', 'F1 Score', 'ROC AUC']
model_list = comparison_df['Model'].tolist()
x          = np.arange(len(model_list))
width      = 0.14

fig, ax = plt.subplots(figsize=(15, 6), facecolor='#ffffff')
ax.set_facecolor('#f6f8fa')
fig.suptitle("Performance Comparison — All 5 Models",
             fontsize=15, fontweight='bold', color='#1f2328', y=1.02)

for i, (metric, mc) in enumerate(zip(metrics, METRIC_COLORS)):
    offset = (i - N_METRICS / 2) * width + width / 2
    bars   = ax.bar(x + offset, comparison_df[metric], width,
                    label=metric, color=mc, alpha=0.85,
                    edgecolor='white', linewidth=0.6, zorder=3)
    for bar in bars:
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.003,
                f"{bar.get_height():.3f}",
                ha='center', va='bottom', fontsize=7,
                rotation=90, color='#24292f', fontweight='bold')

ax.set_xticks(x)
ax.set_xticklabels(model_list, fontsize=10, color='#24292f')
ax.set_ylabel("Score", fontsize=11)
ax.set_ylim(0, 1.18)
ax.legend(loc='lower right')
plt.tight_layout()
plt.savefig("comparison_bar.png", bbox_inches='tight')
plt.show()


# ============================================================
# SECTION 19 : ROC CURVES — ALL MODELS OVERLAID
# ============================================================

fig, ax = plt.subplots(figsize=(8, 7), facecolor='#ffffff')
ax.set_facecolor('#f6f8fa')
ax.set_title("ROC Curves — All Models", fontsize=13, fontweight='bold',
             color='#1f2328', pad=10)

for name, (fpr, tpr, auc_val) in all_roc_data.items():
    mc = MODEL_PALETTE[name]
    ax.plot(fpr, tpr, label=f"{name}  (AUC = {auc_val:.3f})",
            color=mc, linewidth=2.5, zorder=5)
    ax.fill_between(fpr, tpr, alpha=0.05, color=mc)

ax.plot([0, 1], [0, 1], color='#adbac7', linewidth=1.2,
        linestyle='--', label='Random (AUC = 0.500)', zorder=3)
ax.fill_between([0, 1], [0, 1], color='#000000', alpha=0.02)

ax.set_xlabel("False Positive Rate", fontsize=11)
ax.set_ylabel("True Positive Rate",  fontsize=11)
ax.legend(loc='lower right')
ax.set_xlim(-0.01, 1.01)
ax.set_ylim(-0.01, 1.01)
plt.tight_layout()
plt.savefig("roc_curves.png", bbox_inches='tight')
plt.show()


# ============================================================
# SECTION 20 : HEATMAP — METRIC SCORES ACROSS MODELS
# ============================================================

heatmap_df = comparison_df.set_index('Model')[metrics]

dark_blue_cmap = LinearSegmentedColormap.from_list(
    'light_blue', ['#ddf4ff', '#cae8ff', '#79c0ff', '#1f6feb', '#0550ae'], N=256
)

fig, ax = plt.subplots(figsize=(11, 4), facecolor='#ffffff')
ax.set_facecolor('#f6f8fa')

sns.heatmap(
    heatmap_df, annot=True, fmt='.4f', cmap=dark_blue_cmap,
    linewidths=1.2, linecolor='#ffffff', ax=ax,
    vmin=0.75, vmax=1.0,
    annot_kws={'size': 10, 'weight': 'bold'},
    cbar_kws={'shrink': 0.85}
)
ax.set_title("Metric Heatmap — Model Comparison",
             fontsize=13, fontweight='bold', color='#1f2328', pad=10)
ax.tick_params(colors='#24292f', labelsize=10)
ax.set_ylabel("")
ax.figure.axes[-1].yaxis.set_tick_params(labelcolor='#57606a')
plt.tight_layout()
plt.savefig("heatmap_comparison.png", bbox_inches='tight')
plt.show()


# ============================================================
# SECTION 21 : EVALUATION METRICS DASHBOARD
# ============================================================

model_list = comparison_df['Model'].tolist()
palette    = [MODEL_PALETTE[m] for m in model_list]

fig = plt.figure(figsize=(22, 26), facecolor='#ffffff')
fig.suptitle("Evaluation Metrics Dashboard — All 5 Models",
             fontsize=18, fontweight='bold', color='#1f2328', y=0.99)
gs = gridspec.GridSpec(4, 2, figure=fig, hspace=0.55, wspace=0.35)

# ── 21.1  Grouped bar (full width) ───────────────────────────────────────
ax1 = fig.add_subplot(gs[0, :])
ax1.set_facecolor('#f6f8fa')
x = np.arange(len(model_list))
w = 0.15

for i, (metric, mc) in enumerate(zip(metrics, METRIC_COLORS)):
    offset = (i - N_METRICS / 2) * w + w / 2   # fix: uses module-level N_METRICS
    bars   = ax1.bar(x + offset, comparison_df[metric], w,
                     label=metric, color=mc, alpha=0.85,
                     edgecolor='white', linewidth=0.6, zorder=3)
    for bar in bars:
        ax1.text(bar.get_x() + bar.get_width() / 2,
                 bar.get_height() + 0.004,
                 f"{bar.get_height():.3f}",
                 ha='center', va='bottom', fontsize=7,
                 rotation=90, color='#24292f')

ax1.set_xticks(x)
ax1.set_xticklabels(model_list, fontsize=10, color='#24292f')
ax1.set_ylabel("Score", fontsize=11, color='#24292f')
ax1.set_ylim(0, 1.20)
ax1.set_title("All Metrics — Grouped Bar Chart", fontsize=13,
              fontweight='bold', color='#1f2328', pad=10)
ax1.legend(loc='lower right')

# ── 21.2  Per-metric individual bars ─────────────────────────────────────
positions = [(1, 0), (1, 1), (2, 0), (2, 1), (3, 0)]

for (row, col), metric in zip(positions, metrics):
    ax = fig.add_subplot(gs[row, col])
    ax.set_facecolor('#f6f8fa')
    vals = comparison_df[metric].values
    bars = ax.bar(model_list, vals, color=palette,
                  alpha=0.85, edgecolor='white', linewidth=0.6, width=0.52, zorder=3)

    best_idx = int(np.argmax(vals))
    bars[best_idx].set_edgecolor('#d4770a')
    bars[best_idx].set_linewidth(2.0)

    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width() / 2,
                bar.get_height() + 0.002,
                f"{val:.4f}",
                ha='center', va='bottom', fontsize=8.5,
                fontweight='bold', color='#24292f')

    ax.set_title(metric, fontsize=12, fontweight='bold', color='#1f2328', pad=10)
    ax.set_ylabel("Score", fontsize=10, color='#24292f')
    rng = vals.max() - vals.min()
    pad = max(0.02, rng * 0.6)
    ax.set_ylim(max(0, vals.min() - pad), min(1.02, vals.max() + pad * 1.5))
    # fix: set_xticks before set_xticklabels to prevent FixedLocator warning
    ax.set_xticks(range(len(model_list)))
    ax.set_xticklabels(model_list, rotation=20, ha='right', fontsize=8.5, color='#24292f')

# ── 21.3  Radar chart ─────────────────────────────────────────────────────
ax_radar = fig.add_subplot(gs[3, 1], polar=True)
ax_radar.set_facecolor('#f6f8fa')
ax_radar.set_title("Radar Chart — All Models", fontsize=12,
                    fontweight='bold', color='#1f2328', pad=18)
angles = np.linspace(0, 2 * np.pi, len(metrics), endpoint=False).tolist()
angles += angles[:1]

for mname, color in zip(model_list, palette):
    vals = comparison_df[comparison_df['Model'] == mname][metrics].values.flatten().tolist()
    vals += vals[:1]
    ax_radar.plot(angles, vals, color=color, linewidth=2.2, label=mname, zorder=5)
    ax_radar.fill(angles, vals, color=color, alpha=0.10)

ax_radar.set_xticks(angles[:-1])
ax_radar.set_xticklabels(metrics, fontsize=9, color='#24292f')
ax_radar.set_ylim(0, 1)
ax_radar.set_yticks([0.2, 0.4, 0.6, 0.8, 1.0])
ax_radar.set_yticklabels(['0.2', '0.4', '0.6', '0.8', '1.0'],
                          fontsize=7, color='#57606a')
ax_radar.grid(color='#d0d7de', linestyle='--', linewidth=0.8)
ax_radar.spines['polar'].set_color('#d0d7de')
ax_radar.legend(loc='upper right', bbox_to_anchor=(1.45, 1.15), fontsize=8)

plt.savefig("evaluation_metrics_dashboard.png", bbox_inches='tight')
plt.show()
print("Evaluation metrics dashboard saved → evaluation_metrics_dashboard.png")


# ============================================================
# SECTION 22 : BEST MODEL SUMMARY
# ============================================================

best_row = comparison_df.iloc[0]

print("\n" + "=" * 70)
print("BEST MODEL")
print("=" * 70)
print(f"  Name      : {best_row['Model']}")
print(f"  ROC AUC   : {best_row['ROC AUC']:.4f}")
print(f"  Accuracy  : {best_row['Accuracy']:.4f}")
print(f"  F1 Score  : {best_row['F1 Score']:.4f}")
print(f"  Recall    : {best_row['Recall']:.4f}")
print(f"  Precision : {best_row['Precision']:.4f}")


# ============================================================
# SECTION 23 : SAVE RESULTS
# ============================================================

comparison_df.to_csv("5_model_comparison_results.csv", index=False)
print("\nResults saved → 5_model_comparison_results.csv")

print("\n" + "=" * 70)
print("PIPELINE COMPLETED SUCCESSFULLY ✓")
print("=" * 70)
